# DỰ ÁN THỰC THÁI: TIỀN XỬ LÝ DỮ LIỆU VAY VỐN (LOANS DATA PREPROCESSING PROJECT)

Chào mừng bạn đến với dự án thực hành **Loans Data Preprocessing** dựa trên lộ trình Taskade! 
Đây là một bài toán thực tế phổ biến trong ngành Tài chính - Ngân hàng: **Làm sạch và chuẩn hóa dữ liệu hồ sơ vay vốn trước khi đưa vào mô hình AI để đánh giá rủi ro tín dụng (Credit Risk Assessment).**

---

## QUY TRÌNH TIỀN XỬ LÝ TRONG DỰ ÁN
1. **Tải & Khám phá Dữ liệu (Load & Explore Data)**
2. **Xử lý Dữ liệu khuyết (Handling Missing Values)**
3. **Tạo Đặc trưng Mới (Feature Engineering)**
4. **Xử lý Giá trị ngoại lệ (Outliers Detection & Log Transformation)**
5. **Mã hóa Biến phân loại (Categorical Encoding)**
6. **Chuẩn hóa Đặc trưng (Feature Scaling)**
7. **Chia tập dữ liệu (Train - Test Split)**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

# 1. Tải tập dữ liệu loans.csv
df_loans = pd.read_csv('loans.csv')
print("--- 5 HÀNG ĐẦU TIÊN CỦA BẢNG LOANS.CSV ---")
df_loans.head()

--- 5 HÀNG ĐẦU TIÊN CỦA BẢNG LOANS.CSV ---


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0,141.0,360.0,1.0,Urban,Y


In [2]:
# 2. Kiểm tra thông tin tổng quan và missing values
print("--- THÔNG TIN TỔNG QUAN ---")
print(df_loans.info())

print("\n--- SỐ LƯỢNG GIÁ TRỊ THIẾU Ở MỖI CỘT ---")
print(df_loans.isnull().sum())

--- THÔNG TIN TỔNG QUAN ---
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            30 non-null     str    
 1   Gender             29 non-null     str    
 2   Married            30 non-null     str    
 3   Dependents         30 non-null     str    
 4   Education          30 non-null     str    
 5   Self_Employed      26 non-null     str    
 6   ApplicantIncome    30 non-null     int64  
 7   CoapplicantIncome  30 non-null     int64  
 8   LoanAmount         29 non-null     float64
 9   Loan_Amount_Term   29 non-null     float64
 10  Credit_History     28 non-null     float64
 11  Property_Area      30 non-null     str    
 12  Loan_Status        30 non-null     str    
dtypes: float64(3), int64(2), str(8)
memory usage: 3.2 KB
None

--- SỐ LƯỢNG GIÁ TRỊ THIẾU Ở MỖI CỘT ---
Loan_ID              0
Gender               1
Married  

## BƯỚC 1: XỬ LÝ DỮ LIỆU THIẾU (MISSING VALUES IMPUTATION)

- Các cột phân loại (`Gender`, `Married`, `Self_Employed`, `Credit_History`): Điền bằng giá trị xuất hiện nhiều nhất (**Mode**).
- Các cột định lượng (`LoanAmount`, `Loan_Amount_Term`): Điền bằng Trung vị (**Median**) để tránh ảnh hưởng của ngoại lệ.

In [3]:
df_clean = df_loans.copy()

# Điền biến phân loại
cat_cols = ['Gender', 'Married', 'Self_Employed', 'Credit_History', 'Dependents']
for col in cat_cols:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)

# Điền biến định lượng
num_cols = ['LoanAmount', 'Loan_Amount_Term']
for col in num_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

print("Số lượng giá trị missing còn lại:")
print(df_clean.isnull().sum().sum())

Số lượng giá trị missing còn lại:
0


## BƯỚC 2: FEATURE ENGINEERING & LOG TRANSFORMATION

1. **Tạo biến mới:** `TotalIncome = ApplicantIncome + CoapplicantIncome` (Thu nhập tổng của cả hộ gia đình).
2. **Log Transformation:** Dữ liệu thu nhập và số tiền vay thường lệch phải (Right-skewed) với vài khoản vay cực lớn. Biến đổi Log $\ln(X + 1)$ giúp dữ liệu phân phối chuẩn hơn (Normal distribution).

In [4]:
# Tạo thuộc tính TotalIncome
df_clean['TotalIncome'] = df_clean['ApplicantIncome'] + df_clean['CoapplicantIncome']

# Biến đổi Log để giảm lệch (Skewness)
df_clean['LoanAmount_Log'] = np.log(df_clean['LoanAmount'] + 1)
df_clean['TotalIncome_Log'] = np.log(df_clean['TotalIncome'] + 1)

print("--- CÁC CỘT MỚI SAU KHI LOG TRANSFORM ---")
df_clean[['LoanAmount', 'LoanAmount_Log', 'TotalIncome', 'TotalIncome_Log']].head()

--- CÁC CỘT MỚI SAU KHI LOG TRANSFORM ---


,LoanAmount,LoanAmount_Log,TotalIncome,TotalIncome_Log
0,120.0,4.795791,5849,8.674197
1,128.0,4.859812,6091,8.714732
2,66.0,4.204693,3000,8.006701
3,120.0,4.795791,4941,8.505525
4,141.0,4.955827,6000,8.699681


## BƯỚC 3: MÃ HÓA BIẾN PHÂN LOẠI (CATEGORICAL ENCODING)

- `Education`: Ordinal Encoding (`Not Graduate` = 0, `Graduate` = 1).
- `Loan_Status`: Target Encoding (`N` = 0, `Y` = 1).
- `Gender`, `Married`, `Self_Employed`, `Property_Area`: One-Hot Encoding (`get_dummies` hoặc `OneHotEncoder`).

In [5]:
# Label / Ordinal Encoding
df_clean['Education_Enc'] = df_clean['Education'].map({'Not Graduate': 0, 'Graduate': 1})
df_clean['Loan_Status_Enc'] = df_clean['Loan_Status'].map({'N': 0, 'Y': 1})

# One-Hot Encoding cho các biến phân loại danh nghĩa
ohe_cols = ['Gender', 'Married', 'Self_Employed', 'Property_Area', 'Dependents']
df_encoded = pd.get_dummies(df_clean, columns=ohe_cols, drop_first=True)

# Bỏ các cột cũ không cần thiết
cols_to_drop = ['Loan_ID', 'Education', 'Loan_Status', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'TotalIncome']
df_final = df_encoded.drop(columns=cols_to_drop)

print("Kích thước bộ dữ liệu hoàn chỉnh sau Encoding:", df_final.shape)
df_final.head()

Kích thước bộ dữ liệu hoàn chỉnh sau Encoding: (30, 14)


,Loan_Amount_Term,Credit_History,LoanAmount_Log,TotalIncome_Log,Education_Enc,Loan_Status_Enc,Gender_Male,Married_Yes,Self_Employed_Yes,Property_Area_Semiurban,Property_Area_Urban,Dependents_1,Dependents_2,Dependents_3+
0,360.0,1.0,4.795791,8.674197,1,1,True,False,False,False,True,False,False,False
1,360.0,1.0,4.859812,8.714732,1,0,True,True,False,False,False,True,False,False
2,360.0,1.0,4.204693,8.006701,1,1,True,True,True,False,True,False,False,False
3,360.0,1.0,4.795791,8.505525,0,1,True,True,False,False,True,False,False,False
4,360.0,1.0,4.955827,8.699681,1,1,True,False,False,False,True,False,False,False


## BƯỚC 4: FEATURE SCALING & TRAIN-TEST SPLIT

Áp dụng chuẩn quy trình:
1. Tách `X` (Đặc trưng) và `y` (Nhãn mục tiêu `Loan_Status_Enc`).
2. Chia `train_test_split` (75% Train, 25% Test).
3. Áp dụng `StandardScaler` đúng chuẩn tránh Data Leakage!

In [6]:
# Tách X và y
X = df_final.drop(columns=['Loan_Status_Enc'])
y = df_final['Loan_Status_Enc']

# Phân chia Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Chuẩn hóa Feature Scaling bằng StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(" TIỀN XỬ LÝ HOÀN TẤT!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

 TIỀN XỬ LÝ HOÀN TẤT!
X_train_scaled shape: (22, 13)
X_test_scaled shape: (8, 13)
